# Week 1 - Video -> Browsable 3D Room (Person A pipeline)
**Notebook settings (right sidebar):** Accelerator = *GPU T4 x2* - Internet = *On* - Persistence = *Files only*.

Each stage below is a **separate process** (`!python ...`) so VRAM is released after every stage (PDF 4.3).
Run the cells top to bottom the first time. Re-runs can skip setup.

In [1]:
# 0. Get the code. EITHER clone your repo ...
#REPO = "https://github.com/<your-team>/<your-repo>.git"      # <- edit
# !git clone $REPO /kaggle/working/repo
# ... OR attach the repo as a Kaggle Dataset and copy it:
!cp -r /kaggle/input/datasets/shahdelsawy/full-w12/week1_pipeline /kaggle/working/
%cd /kaggle/working/week1_pipeline

/kaggle/working/week1_pipeline


In [2]:
# 1. One-time setup: ffmpeg, VGGT, pycolmap 4 (GLOMAP built in, no conda), gsplat.  ~5-15 min the first time.
#    (the old route with the conda colmap/glomap binaries is still there: `setup_kaggle.sh all`, then --backend cli)
!bash scripts/setup_kaggle.sh fast

=== base python deps ===
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
=== VGGT ===
Cloning into '/kaggle/working/vggt'...
remote: Enumerating objects: 199, done.
remote: Counting objects: 100% (199/199), done.
remote: Compressing objects: 100% (184/184), done.
remote: Total 199 (delta 6), reused 187 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (199/199), 62.06 MiB | 52.61 MiB/s, done.
Resolving deltas: 100% (6/6), done.
VGGT import OK, torch 2.11.0+cu130
=== pycolmap 4 (GLOMAP built in, GPU SIFT via CUDA) - isolated folder ===
pycolmap 4.2.1 | cuda: True | global_mapping: True
=== gsplat + examples ===
python: /usr/local/bin/python
gsplat version: 1.5.3
Note: switching to '937e29912570c372bed6747a5c9bf85fed877bae'.
You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.
If you want to c

In [3]:
# 2. Environment for this session
import os, re, shutil
for line in open("/kaggle/working/env.sh"):
    m = re.match(r"export (\w+)=(.*)", line.strip())
    if m: os.environ[m.group(1)] = os.path.expandvars(m.group(2))
# safety: the conda env's bare python must never shadow the system python
os.environ["PATH"] = ":".join(x for x in os.environ["PATH"].split(":") if x and "/kaggle/working/cm/bin" not in x and "$PATH" not in x)
print("python :", shutil.which("python"), "| colmap:", shutil.which("colmap"), "| glomap:", shutil.which("glomap"))
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"          # T4
os.environ["W1_ROOT"] = "/kaggle/working/workspace"   # everything is written here (persisted in Output)
os.environ["W1_PROJECT"] = "demo"
VIDEO = "/kaggle/input/datasets/shahdelsawy/room-video/video.mp4"    # <- edit: 2-5 min, one room, slow motion
!nvidia-smi --query-gpu=name,memory.total --format=csv

python : /usr/local/bin/python | colmap: None | glomap: None
name, memory.total [MiB]
Tesla T4, 15360 MiB


## Stage 1 - frames (ffmpeg, 1-4 fps, long side <= 960, blur filter)

In [4]:
!python stages/s1_extract_frames.py --video $VIDEO --fps auto --long-side 960

[s1] 61 frames @ 1.0 fps -> kept 61 (0 blurry dropped), size 960x540


## Stage 2 - poses. Run BOTH for the required VGGT-vs-GLOMAP experiment (same frames)

In [5]:
# VGGT: fp16 on T4, chunks of 32 frames with 8 shared frames; also measure VRAM vs frames-per-chunk
!python stages/s2a_poses_vggt.py --chunk-size 32 --overlap 8 --memory-sweep 8,16,32,48

[vggt] device=cuda dtype=torch.float16 frames=61
[vggt] memory sweep: [{'frames': 8, 'seconds': 7.05, 'peak_alloc_mb': 8499.4}, {'frames': 16, 'seconds': 3.46, 'peak_alloc_mb': 8726.2}, {'frames': 32, 'seconds': 9.2, 'peak_alloc_mb': 9162.3}, {'frames': 48, 'seconds': 17.59, 'peak_alloc_mb': 9594.6}]
[vggt] 3 chunk(s): [(0, 32), (24, 56), (29, 61)]
[vggt] done: 61 cameras, 300000 points -> /kaggle/working/workspace/demo/poses/vggt


In [6]:
# GLOMAP via pycolmap 4 (auto-detected through W1_PYCOLMAP_DIR): GPU SIFT -> hybrid matching -> global mapping
# -> checks (falls back to incremental mapping if the result looks unreliable) -> prune bad frames -> undistort
!python stages/s2b_poses_glomap.py
# old route instead:  !python stages/s2b_poses_glomap.py --backend cli --overlap 12

[glomap] backend: pycolmap
[pycolmap] pycolmap 4.2.1 (cuda=True)
I20261113 02:27:42.836555 138807731025600 feature_extraction.cc:498] === Feature extraction ===
I20261113 02:27:42.837376 138807689062080 sift.cc:758] Creating SIFT GPU feature extractor
I20261113 02:27:42.837462 138807680669376 sift.cc:758] Creating SIFT GPU feature extractor
I20261113 02:27:43.335671 138807672276672 feature_extraction.cc:270] Processed file [1/61]
I20261113 02:27:43.335716 138807672276672 feature_extraction.cc:273]   Name:            000001.jpg
I20261113 02:27:43.335724 138807672276672 feature_extraction.cc:282]   Dimensions:      960 x 540
I20261113 02:27:43.335731 138807672276672 feature_extraction.cc:285]   Camera:          #1 - SIMPLE_RADIAL
I20261113 02:27:43.335739 138807672276672 feature_extraction.cc:289]   Focal Length:    700.80px
I20261113 02:27:43.335753 138807672276672 feature_extraction.cc:294]   Features:        3271 (SIFT)
I20261113 02:27:43.369954 138807672276672 feature_extraction.cc:2

In [7]:
# Comparison report + pick the method used for the project (writes scene/)
!python stages/s3_compare_poses.py --choose auto
from IPython.display import Markdown, Image, display
display(Markdown(open(os.environ["W1_ROOT"] + "/demo/metrics/pose_comparison.md").read()))

# VGGT vs GLOMAP
Frames fed to both methods: **61**
| method | registered | runtime (s) | peak VRAM (MiB) | points3D | path length |
|---|---|---|---|---|---|
| vggt | 61 | 118.23 | 11363.0 | 300,000 | 3.212 |
| glomap | 61 | 37.61 | 543.0 | 11,286 | 33.608 |
Trajectory agreement (Sim3-aligned, 61 common frames): RMSE = 0.70% of scene extent.
![visual check](pose_comparison.png)
## Draft justification (edit before submitting)
We chose GLOMAP for the project because both register all frames and runtimes are comparable, so the more accurate classical baseline wins. VGGT: 61 frames, 118s, 11363 MiB peak VRAM. GLOMAP: 61 frames, 38s, 543 MiB peak VRAM.
[compare] scene/ <- glomap


# VGGT vs GLOMAP

Frames fed to both methods: **61**

| method | registered | runtime (s) | peak VRAM (MiB) | points3D | path length |
|---|---|---|---|---|---|
| vggt | 61 | 118.23 | 11363.0 | 300,000 | 3.212 |
| glomap | 61 | 37.61 | 543.0 | 11,286 | 33.608 |

Trajectory agreement (Sim3-aligned, 61 common frames): RMSE = 0.70% of scene extent.

![visual check](pose_comparison.png)

## Draft justification (edit before submitting)

We chose GLOMAP for the project because both register all frames and runtimes are comparable, so the more accurate classical baseline wins. VGGT: 61 frames, 118s, 11363 MiB peak VRAM. GLOMAP: 61 frames, 38s, 543 MiB peak VRAM.


In [8]:
# OPTIONAL but recommended once per scene: which pose set renders held-out frames better? (trains twice at 7k iters, ~2 x 15 min;
# your current choice is restored afterwards; table -> metrics/pose_method_psnr.md). Skip when poses already agree (see the Sim(3) line above).
# !python tools/compare_pose_methods.py --iters 7000

## Stage 4 - 3D Gaussian Splatting (quick = 7k iters to confirm, then full = 30k). The 30k run is only worth it for the final output: on the 61-frame test it scored the same PSNR as 7k

In [9]:
!python stages/s4_train_3dgs.py --trainer gsplat --mode quick

[train] created /kaggle/working/gsplat/examples/datasets/__init__.py (avoids clash with Hugging Face 'datasets')
[train] patched /usr/local/lib/python3.13/dist-packages/pycolmap/scene_manager.py for numpy>=2 (np.uint64(-1))
[train] gsplat 7000 iters in 15.2 min; 1326004 Gaussians; held-out {'psnr': 25.91569709777832, 'ssim': 0.8792511820793152, 'lpips': 0.0817364752292633}


In [10]:
# quality run (about 30k iters; check the quick run looked sane first!)
!python stages/s4_train_3dgs.py --trainer gsplat --mode full

[train] gsplat 30000 iters in 41.1 min; 1488141 Gaussians; held-out {'psnr': 26.776355743408203, 'ssim': 0.8870153427124023, 'lpips': 0.06398005038499832}


## Stage 5 - export for the web viewer, then the generated deliverables

In [11]:
!python stages/s5_export_splat.py
!python tools/make_reports.py
# Week 2 inputs: week2/inputs.json (+ sparse depth, and per-frame depth rendered from the splat when the GPU is free)
!python stages/s6_prepare_week2.py
!python tools/make_reports.py
!cat docs/generated/VRAM_RUNTIME_TABLE.md

[export] 1,059,412 Gaussians; ply 238.44 MB, splat 32.33 MB
wrote docs/generated/VRAM_RUNTIME_TABLE.md, docs/generated/WEEK2_HANDOFF.md, /kaggle/working/workspace/demo/manifest.json
[week2] wrote week2/inputs.json: 61 frames, depth=splat_rendered, edit space identical between frames/ and scene/images
wrote docs/generated/VRAM_RUNTIME_TABLE.md, docs/generated/WEEK2_HANDOFF.md, /kaggle/working/workspace/demo/manifest.json
# VRAM and runtime per stage
| stage | wall time (s) | peak VRAM, nvidia-smi (MiB) | peak VRAM, torch alloc (MiB) | notes |
|---|---|---|---|---|
| 1. Frame extraction (ffmpeg + blur filter) | 123.7 | 3 | 0 | frames_kept=61 |
| 2a. Poses: VGGT (chunked, fp16) | 118.2 | 11363 | 9595 | registered_frames=61, chunks=3, dtype=torch.float16 |
| 2b. Poses: GLOMAP (features + seq. matching + mapper) | 37.6 | 543 | 0 | registered_frames=61 |
| 4. 3DGS training | 2465.7 | 3653 | 0 | num_gaussians=1488141, iters=30000 |
| 5. Export (.ply -> .splat) | 2.9 | 3 | 0 | num_gaussians=10

In [12]:
# 6. Persist what matters from /kaggle/working before the session ends (PDF 4.3)
!cd $W1_ROOT/demo && zip -qr /kaggle/working/demo_export.zip export manifest.json metrics scene frames/frames_index.json
!ls -lh /kaggle/working/demo_export.zip
# Week 2 inputs: everything listed in week2/inputs.json -> "required_files" (frames, scene, week2/, checkpoint in gs/).
import json, subprocess
req = json.load(open(os.environ["W1_ROOT"] + "/demo/week2/inputs.json"))["required_files"]
subprocess.run(["zip", "-qr", "/kaggle/working/demo_week2_inputs.zip", *req], cwd=os.environ["W1_ROOT"] + "/demo", check=True)
!ls -lh /kaggle/working/demo_week2_inputs.zip

-rw-r--r-- 1 root root 235M Nov 13 03:25 /kaggle/working/demo_export.zip
-rw-r--r-- 1 root root 329M Nov 13 03:25 /kaggle/working/demo_week2_inputs.zip


In [13]:
import os
from IPython.display import FileLink, display

os.chdir('/kaggle/working')

print(os.listdir('.'))

['vggt', '.virtual_documents', 'week1_pipeline', 'tools', 'demo_week2_inputs.zip', 'workspace', 'env.sh', 'demo_export.zip', 'gsplat']


In [14]:
display(FileLink('demo_export.zip'))

/kaggle/working/demo_export.zip

In [15]:
display(FileLink('demo_week2_inputs.zip'))

/kaggle/working/demo_week2_inputs.zip

### One-shot alternative
```
!python run_pipeline.py --video $VIDEO --project demo --method both --train-mode quick   # add --glomap-backend cli to use the colmap/glomap binaries
```
With the backend: add `--backend-url https://<your>.trycloudflare.com --task-id <id>` and the real stage progress is POSTed to `/tasks/{id}/progress`.